In [1]:
import requests
import json
import glob
from openai import OpenAI

In [ ]:
# ClickHouse connection
CLICKHOUSE_HOST = "http://localhost:8123"
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = "model-gateway-1"
CLICKHOUSE_DATABASE = "modelgateway1"

# Inference endpoint
INFERENCE_BASE_URL = ""
INFERENCE_API_KEY = ""
INFERENCE_MODEL = ""

def query_clickhouse(sql):
    """Execute a query against ClickHouse"""
    url = f"{CLICKHOUSE_HOST}/?default_format=JSON&database={CLICKHOUSE_DATABASE}"
    auth = (CLICKHOUSE_USER, CLICKHOUSE_PASSWORD)
    response = requests.post(url, data=sql, auth=auth)
    response.raise_for_status()
    return response.json()

In [3]:
# Explore response_payload column structure
schema = query_clickhouse("DESCRIBE TABLE query_logs")
for col in schema['data']:
    print(f"{col['name']:40s} {col['type']}")

id                                       UInt64
key_id                                   String
model                                    String
content                                  String
request_payload                          JSON
response_payload                         JSON
duration_first_token                     UInt64
duration_completed                       UInt64
input_token                              UInt64
output_token                             UInt64
cache_token                              UInt64


In [4]:
# Peek at payload
sample = query_clickhouse("""
SELECT id, response_payload
FROM query_logs
WHERE length(toString(response_payload)) > 100
LIMIT 1
""")
row = sample['data'][0]
print("id:", row['id'])
print("response_payload type:", type(row['response_payload']))
print("response_payload:")
print(json.dumps(row['response_payload'], indent=2)[:2000])

id: 3331475046240619666
response_payload type: <class 'dict'>
response_payload:
{
  "content": [
    {
      "signature": "564297ec271df5835be2c3394bb18d32b07c3874fe002249bee5471c68a7e8a4",
      "thinking": "Let me try to run `go build` to verify the code compiles correctly.",
      "type": "thinking"
    },
    {
      "type": "tool_use"
    }
  ],
  "id": "05f6d16551278659b0dec0b514d65cd5",
  "model": "MiniMax-M2.5",
  "role": "assistant",
  "stop_reason": "tool_use",
  "type": "message",
  "usage": {
    "cache_read_input_tokens": 47830,
    "input_tokens": 211,
    "output_tokens": 87
  }
}


In [5]:
# Load all 01-*.json range files
range_files = sorted(glob.glob("01-*.json"))
print(f"Found range files: {range_files}")

ranges = []
for f in range_files:
    with open(f) as fp:
        data = json.load(fp)
    ranges.append({"file": f, "startId": data["startId"], "endId": data["endId"]})
    print(f"  {f}: {data['startId']} -> {data['endId']}")

Found range files: ['01-backend-connector-fix-glm.json', '01-backend-connector-fix-minimax.json', '01-frontend-button-fix-glm.json', '01-frontend-button-fix-minimax.json']
  01-backend-connector-fix-glm.json: 3331470522608255031 -> 3331471570982930244
  01-backend-connector-fix-minimax.json: 3331472113092528050 -> 3331475149932203281
  01-frontend-button-fix-glm.json: 3331229494005465783 -> 3331229943613884106
  01-frontend-button-fix-minimax.json: 3331211524202037408 -> 3331212237959333020


In [6]:
# Query ClickHouse for all logs within the 01-*.json ranges
range_clauses = " OR ".join(
    f"(id >= {r['startId']} AND id <= {r['endId']})"
    for r in ranges
)

result = query_clickhouse(f"""
SELECT id, model, toString(request_payload) AS request_payload
FROM query_logs
WHERE {range_clauses}
ORDER BY id ASC
""")
logs = result['data']
# Parse request_payload strings back to dicts
for r in logs:
    if isinstance(r['request_payload'], str):
        try:
            r['request_payload'] = json.loads(r['request_payload'])
        except json.JSONDecodeError:
            r['request_payload'] = {}

print(f"Fetched {len(logs)} logs")

Fetched 119 logs


In [7]:
def extract_last_user_text(request_payload):
    """Extract last user message text for preview"""
    messages = request_payload.get("messages", [])
    for msg in reversed(messages):
        if msg.get("role") == "user":
            content = msg.get("content", "")
            if isinstance(content, str):
                return content
            if isinstance(content, list):
                for block in content:
                    if isinstance(block, dict) and block.get("type") == "text":
                        return block.get("text", "")
    return ""

# Preview
for r in logs[:3]:
    msgs = r['request_payload'].get("messages", [])
    preview = extract_last_user_text(r['request_payload'])
    print(f"=== id={r['id']} ===")
    print(f"total messages: {len(msgs)}, will send last 5")
    print(f"last user msg: {preview[:300]}")

=== id=3331211524202037408 ===
total messages: 1, will send last 5
last user msg: <system-reminder>
As you answer the user's questions, you can use the following context:
# currentDate
Today's date is 2026-03-03.

      IMPORTANT: this context may or may not be relevant to your tasks. You should not respond to this context unless it is highly relevant to your task.
</system-remin
=== id=3331211546561872109 ===
total messages: 3, will send last 5
last user msg: <system-reminder>
As you answer the user's questions, you can use the following context:
# currentDate
Today's date is 2026-03-03.

      IMPORTANT: this context may or may not be relevant to your tasks. You should not respond to this context unless it is highly relevant to your task.
</system-remin
=== id=3331211621195317541 ===
total messages: 5, will send last 5
last user msg: <system-reminder>
As you answer the user's questions, you can use the following context:
# currentDate
Today's date is 2026-03-03.

      IMPORTANT: thi

In [8]:
# Set up client
client = OpenAI(
    base_url=INFERENCE_BASE_URL,
    api_key=INFERENCE_API_KEY,
)

CATEGORIES = ["explore", "implement", "test", "debug", "prompt"]

SYSTEM_PROMPT = """You are a classifier for AI coding assistant requests.
Given the request payload JSON, classify the AI's current action into exactly one of these categories:

- explore: Reading/exploring code, understanding architecture, listing files, examining structure, gathering information
- implement: Writing new code, implementing features, creating files, refactoring existing code
- test: Writing tests, running tests, verifying functionality, checking test results
- debug: Fixing bugs, inv0estigating errors, tracing issues, diagnosing problems
- prompt: Telling the user 
" clarifying question, requesting more information, waiting for user input

Respond with ONLY the single category word, nothing else."""

def classify_response(request_payload):
    payload = {
        k: (v[-5:] if k == "messages" else v)
        for k, v in request_payload.items()
        if k != "tools"
    }
    payload_json = json.dumps(payload, ensure_ascii=False)

    resp = client.chat.completions.create(
        model=INFERENCE_MODEL,
        max_tokens=512,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this AI coding assistant request:\n\n```{payload_json}``"}
        ]
    )
    category = resp.choices[0].message.content.strip().lower()
    return category

print("Classifier ready.")

Classifier ready.


In [9]:
# Load existing categories.json
CATEGORIES_PATH = "categories.json"
try:
    with open(CATEGORIES_PATH) as f:
        categories = json.load(f)
    print(f"Loaded {len(categories)} existing entries from {CATEGORIES_PATH}")
except (FileNotFoundError, json.JSONDecodeError):
    categories = {}
    print("No existing categories.json, starting fresh")

pending = [r for r in logs if str(r['id']) not in categories]
print(f"Pending: {len(pending)} / {len(logs)} logs need classification")

Loaded 72 existing entries from categories.json
Pending: 54 / 119 logs need classification


In [10]:
for i, r in enumerate(pending):
    log_id = str(r['id'])
    category = classify_response(r['request_payload'])
    categories[log_id] = category
    print(f"[{i+1:2d}/{len(pending)}] {log_id}  -> {category}")

print(f"\nDone. Total classified: {len(categories)} logs.")

[ 1/54] 3331472113092528050  -> implement
[ 2/54] 3331472221326540825  -> implement
[ 3/54] 3331472269250658378  -> explore
[ 4/54] 3331472312770756776  -> explore
[ 5/54] 3331472358773883082  -> explore
[ 6/54] 3331472391342653752  -> explore
[ 7/54] 3331472431675081046  -> explore
[ 8/54] 3331472457520382354  -> explore
[ 9/54] 3331472502143582706  -> explore
[10/54] 3331472556984107583  -> explore
[11/54] 3331472598860038761  -> explore
[12/54] 3331472641037959867  -> explore
[13/54] 3331472715377804031  -> explore
[14/54] 3331472734281532196  -> explore
[15/54] 3331472752455451516  -> explore
[16/54] 3331472771535340475  -> explore
[17/54] 3331472847208973265  -> explore
[18/54] 3331472959532434495  -> explore
[19/54] 3331472985142854757  -> 
[20/54] 3331473036971869328  -> implement
[21/54] 3331473098066101446  -> explore
[22/54] 3331473127384286527  -> explore
[23/54] 3331473174180136309  -> explore
[24/54] 3331473264827434374  -> explore
[25/54] 3331473345953662431  -> explore
[

In [11]:
# Summary
from collections import Counter
counts = Counter(categories.values())
print("Category distribution:")
for cat, n in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "#" * n
    print(f"  {cat:12s}: {n:3d}  {bar}")

Category distribution:
  explore     :  58  ##########################################################
  debug       :  38  ######################################
  implement   :  23  #######################
              :   6  ######
  test        :   1  #


In [12]:
# Save to categories.json
output_path = "categories.json"
with open(output_path, "w") as f:
    json.dump(categories, f, indent=2)

print(f"Saved {len(categories)} entries to {output_path}")
print(json.dumps(categories, indent=2))

Saved 126 entries to categories.json
{
  "3331228742562349063": "debug",
  "3331228785470079068": "debug",
  "3331228814037483649": "debug",
  "3331228844819480813": "explore",
  "3331228875970576640": "explore",
  "3331228913799004512": "explore",
  "3331228966815007110": "implement",
  "3331229494005465783": "debug",
  "3331229522002445051": "debug",
  "3331229556513178391": "debug",
  "3331229591732749137": "explore",
  "3331229624217633719": "explore",
  "3331229654710223835": "explore",
  "3331229696636486658": "explore",
  "3331229728752272500": "explore",
  "3331229758695408788": "explore",
  "3331229799317243135": "explore",
  "3331229812776764714": "explore",
  "3331229823388353899": "explore",
  "3331229832443856318": "explore",
  "3331229840522085879": "explore",
  "3331229847648208403": "explore",
  "3331229861812373103": "explore",
  "3331229911573595814": "implement",
  "3331229943613884106": "implement",
  "3331211524202037408": "debug",
  "3331211546561872109": "debug",